# 真实本地模型：ITI 逐头探针转向（拟合与运行）

先为每个注意力头训练逻辑回归探针，按验证准确率选出 top-K 个“真实性方向”并保存工件；生成时沿 α·σ·θ 偏移这些头的输出。

## 真实集成前提

安装：`python3 -m pip install -e ".[hf,iti]"`。拟合需要 `prompt/response/label` 标注数据（label 1=真实、0=虚假）；`bake=True` 在 v0.1 明确不支持。

代码单元没有预执行。填写本地模型路径或 API 凭据后，从上到下运行；不要将 API key 保存到 notebook。缓解会改变模型输出，务必同时阅读基线与缓解后的回答。

## 参数说明

- `top_k: int = 48`：干预的注意力头数（论文默认 48）。
- `alpha: float = 15.0`：偏移强度（以投影标准差为单位，论文默认 15）。
- `chat=True`：标注数据按 chat 模板渲染。
- `MAX_NEW_TOKENS: int = 64`：生成长度。

## 步骤 1：配置真实模型与标注数据

In [ ]:
from pathlib import Path
import sys
repo_root = Path.cwd()
if not (repo_root / "src" / "hallucination").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

LOCAL_MODEL = "/absolute/path/to/your/causal-lm"   # e.g. a local Qwen2.5-1.5B-Instruct directory
DEVICE = "cpu"     # "cuda" when a GPU is available
DTYPE = "auto"
MAX_NEW_TOKENS = 64
if not Path(LOCAL_MODEL).is_dir():
    raise ValueError("Set LOCAL_MODEL to an existing local model directory (download it first).")

import json
DATA = repo_root / "examples" / "mitigation" / "data" / "iti_demo.jsonl"
ARTIFACT = repo_root / "runs" / "iti_demo.json"
samples = [json.loads(line) for line in DATA.read_text(encoding="utf-8").splitlines() if line.strip()]
print("samples:", len(samples), "| first keys:", sorted(samples[0]))

## 步骤 2：拟合并保存探针工件

In [ ]:
from hallucination.mitigation.core.model_config import ModelConfig
from hallucination.mitigation.engine.hf import HFEngine
from hallucination.mitigation.methods.iti import ITIConfig, fit_iti
engine = HFEngine.from_pretrained(ModelConfig(model=LOCAL_MODEL, device=DEVICE, local_files_only=True))
artifact = fit_iti(engine, samples, ITIConfig(top_k=48, alpha=15.0, chat=True))
ARTIFACT.parent.mkdir(exist_ok=True)
artifact.save(str(ARTIFACT))
print("artifact:", ARTIFACT, "| heads:", len(artifact.heads), "| best accuracy:", artifact.metadata.get("best_accuracy"))

## 步骤 3：加载探针交互运行

In [ ]:
from hallucination.mitigation import Mitigator
mitigator = Mitigator(
    method="iti",
    model=LOCAL_MODEL,
    device=DEVICE,
    dtype=DTYPE,
    local_files_only=True,
    probe=str(ARTIFACT),
    baseline=True,
)
prompt = input("Prompt：").strip()
report = mitigator.mitigate(prompt, max_new_tokens=MAX_NEW_TOKENS)
print("baseline :", report.baseline)
print("mitigated:", report.response)
print("warnings:", report.warnings)

## 结果解读与限制

仓库自带的 iti_demo.jsonl 只有十余条样本，探针准确率接近随机——真实使用必须准备数百条以上带真/假标签的问答对；探针工件与模型结构绑定。